### Used Car Price Prediction

In [38]:
#importing libraries

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

In [13]:
#importing dataset into dataframe

df = pd.read_csv("data\\cardekho_imputated.csv")
df.head()

,Unnamed: 0,car_name,brand,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats,selling_price
0,0,Maruti Alto,Maruti,Alto,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5,120000
1,1,Hyundai Grand,Hyundai,Grand,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5,550000
2,2,Hyundai i20,Hyundai,i20,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5,215000
3,3,Maruti Alto,Maruti,Alto,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5,226000
4,4,Ford Ecosport,Ford,Ecosport,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5,570000


In [14]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 15411 entries, 0 to 15410
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Unnamed: 0         15411 non-null  int64  
 1   car_name           15411 non-null  str    
 2   brand              15411 non-null  str    
 3   model              15411 non-null  str    
 4   vehicle_age        15411 non-null  int64  
 5   km_driven          15411 non-null  int64  
 6   seller_type        15411 non-null  str    
 7   fuel_type          15411 non-null  str    
 8   transmission_type  15411 non-null  str    
 9   mileage            15411 non-null  float64
 10  engine             15411 non-null  int64  
 11  max_power          15411 non-null  float64
 12  seats              15411 non-null  int64  
 13  selling_price      15411 non-null  int64  
dtypes: float64(2), int64(6), str(6)
memory usage: 1.6 MB


In [15]:
#analyzing features and data cleaning

#dropping columns "Unnamed: 0", "brand" and "car_name"

df.drop(columns=["Unnamed: 0", "brand", "car_name"], inplace=True) 

In [16]:
df['model'].nunique()

120

In [17]:
df.columns

Index(['model', 'vehicle_age', 'km_driven', 'seller_type', 'fuel_type',
       'transmission_type', 'mileage', 'engine', 'max_power', 'seats',
       'selling_price'],
      dtype='str')

In [18]:
#getting all different kind of features

num_features = [col for col in df.columns if df[col].dtype !="str"]
cat_features = [col for col in df.columns if df[col].dtype =="str"]
continuous_num_features = [feature for feature in num_features if df[feature].nunique() >= 25]
discrete_num_features = [feature for feature in num_features if df[feature].nunique() <= 25]

In [19]:
#splitting independent and dependent features

x = df.drop(columns=['selling_price'])
y = df['selling_price']

In [20]:
#training and testing data

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=12)

In [21]:
#categorical features encoding and scaling

#applying labelencoding on "model" feature since it has 120 unique values

label_encoder = LabelEncoder()
x['model']=label_encoder.fit(x['model'])
x_train['model']=label_encoder.transform(x_train['model'])
x_test['model']=label_encoder.transform(x_test['model'])


In [29]:
#applying onehotencoding on features "seller_type", "fuel_type" and "transmission_type"
cat_col = ["seller_type", "fuel_type", "transmission_type"]
preprocessor = ColumnTransformer(
    [
        ("one_hot_encoder", OneHotEncoder(drop="first"), cat_col),
        ("standard_scaler", StandardScaler(), num_features[:-1])],
        remainder="passthrough"
)

In [30]:
#preprocessing train and test data

x_train = preprocessor.fit_transform(x_train)
x_test = preprocessor.transform(x_test)

In [41]:
#model training

def model_evaluation(y_test, y_pred):
    r2_value = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)

    return r2_value, mae, mse

models = {"linear_regression": LinearRegression(),
          "ridge": Ridge(),
          "lasso": Lasso(),
          "k_neighbors": KNeighborsRegressor(),
          "decision_tree": DecisionTreeRegressor(),
          "random_forest": RandomForestRegressor()}

for item in range(len(models)):
    model = list(models.values())[item]

    #training
    model.fit(x_train, y_train)

    #prediction
    y_train_pred = model.predict(x_train)
    y_test_pred = model.predict(x_test)

    #evaluation
    train_r2_value, train_mae, train_mse = model_evaluation(y_train, y_train_pred)
    test_r2_value, test_mae, test_mse = model_evaluation(y_test, y_test_pred)


    current_model = list(models)[item]

    print(f"Model: {current_model}\n")

    print("Model performance on training data")
    print(f"R_Square: {train_r2_value}")
    print(f"MAE: {train_mae:.2f}")
    print(f"MSE: {train_mse:.2f}\n")

    print("Model performance on testing data")
    print(f"R_Square: {test_r2_value}")
    print(f"MAE: {test_mae:.2f}")
    print(f"MSE: {test_mse:.2f}\n\n")


Model: linear_regression

Model performance on training data
R_Square: 0.6612307930155471
MAE: 259297.47
MSE: 235618827537.69

Model performance on testing data
R_Square: 0.5500143903754567
MAE: 269914.22
MSE: 546281185608.83


Model: ridge

Model performance on training data
R_Square: 0.6612292911079227
MAE: 259261.05
MSE: 235619872135.92

Model performance on testing data
R_Square: 0.5499929656724657
MAE: 269833.34
MSE: 546307195134.25


Model: lasso

Model performance on training data
R_Square: 0.6612307830697265
MAE: 259295.34
MSE: 235618834455.15

Model performance on testing data
R_Square: 0.5500154149982861
MAE: 269907.66
MSE: 546279941719.77


Model: k_neighbors

Model performance on training data
R_Square: 0.9223601184914927
MAE: 83268.38
MSE: 53999647766.21

Model performance on testing data
R_Square: 0.7074441462046763
MAE: 119291.61
MSE: 355161932403.71


Model: decision_tree

Model performance on training data
R_Square: 0.9994544497726245
MAE: 4945.90
MSE: 379437983.99

Mo

KNeighbors, DecisionTree and RanodmForestRegressor are performing well on given dataset.

In [ ]:
#hyperparameter tuning on these models

knn_params={}
